In [1]:
# Add beh_ephys_analysis to path for imports
import os
import sys

# Get current directory
current_dir = os.getcwd()
print(f"Current directory: {current_dir}")

# Try to find beh_ephys_analysis
beh_ephys_path = None

if current_dir.endswith('/code') or current_dir == '/code':
    test_path = os.path.join(current_dir, 'beh_ephys_analysis')
    if os.path.exists(os.path.join(test_path, 'utils')):
        beh_ephys_path = test_path

if beh_ephys_path and os.path.exists(os.path.join(beh_ephys_path, 'utils')):
    if beh_ephys_path in sys.path:
        sys.path.remove(beh_ephys_path)
    sys.path.insert(0, beh_ephys_path)
    print(f"✓ Successfully added to path: {beh_ephys_path}")
else:
    print(f"✗ ERROR: Could not find beh_ephys_analysis/utils")
    raise ImportError("Cannot locate beh_ephys_analysis module")

# Now do the imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils.beh_functions import *
from utils.pupil_utils import load_pupil
from aind_dynamic_foraging_data_utils.nwb_utils import load_nwb_from_filename
from aind_dynamic_foraging_basic_analysis.plot.plot_foraging_session import plot_foraging_session, plot_foraging_session_nwb
from aind_dynamic_foraging_basic_analysis.licks.lick_analysis import plot_lick_analysis, cal_metrics, plot_met, load_data
from harp.clock import decode_harp_clock, align_timestamps_to_anchor_points
from open_ephys.analysis import Session
import datetime
from aind_ephys_rig_qc.temporal_alignment import search_harp_line
from matplotlib.gridspec import GridSpec
import json
import itertools

print("✓ All imports successful!")
%matplotlib inline

Current directory: /code
✓ Successfully added to path: /code/beh_ephys_analysis
✓ All imports successful!


In [2]:
"""
Comprehensive test for build_combined_nwb() - validates columns, names, descriptions, and values

Checks:
1. All source columns are included with correct mapped names
2. Descriptions match column_names_description.json
3. Values match between source and combined NWB
4. Data modality tracking
"""
import json
import numpy as np

sys.path.insert(0, '/root/capsule/code/data_management')
from build_merged_nwb import build_combined_nwb, merge_unit_tables

# Load mappings and descriptions
with open('/root/capsule/code/data_management/column_names_map.json', 'r') as f:
    COLUMN_MAP = json.load(f)
with open('/root/capsule/code/data_management/column_names_description.json', 'r') as f:
    COLUMN_DESC = json.load(f)

# Predefined NWB columns that are handled separately (not via add_unit_column)
PREDEFINED_COLS = ['spike_times', 'electrodes', 'obs_intervals', 'electrode_group']

def compare_values(source_val, nwb_val, col_name):
    """Compare two values and return if they're equal and any differences"""
    if source_val is None or (isinstance(source_val, float) and pd.isna(source_val)):
        # Source is None/NaN
        if isinstance(nwb_val, np.ndarray) and len(nwb_val) == 0:
            return True, None  # None -> empty array is OK
        if isinstance(nwb_val, float) and np.isnan(nwb_val):
            return True, None  # NaN preserved
        return False, f"Source is None but NWB is {type(nwb_val).__name__}"
    
    if isinstance(source_val, np.ndarray) and isinstance(nwb_val, np.ndarray):
        if source_val.shape != nwb_val.shape:
            return False, f"Shape mismatch: {source_val.shape} vs {nwb_val.shape}"
        if not np.allclose(source_val, nwb_val, equal_nan=True):
            return False, f"Array values differ"
        return True, None
    
    if isinstance(source_val, (int, float, np.integer, np.floating)) and isinstance(nwb_val, (int, float, np.integer, np.floating)):
        if pd.isna(source_val) and pd.isna(nwb_val):
            return True, None
        if np.isclose(source_val, nwb_val, equal_nan=True):
            return True, None
        return False, f"Value mismatch: {source_val} vs {nwb_val}"
    
    if source_val == nwb_val:
        return True, None
    
    return False, f"Value mismatch: {source_val} vs {nwb_val}"

def test_session_comprehensive(session_id, data_type='curated'):
    """Comprehensive validation of combined NWB"""
    print(f"\n{'#'*80}")
    print(f"# {session_id}")
    print(f"{'#'*80}\n")
    
    # Get source data
    session_tbl = get_session_tbl(session_id)
    merged_units = merge_unit_tables(session_id, data_type)
    
    # Build combined NWB
    save_path, nwb, modalities = build_combined_nwb(session_id, data_type, save_file=None)
    # build_combined_nwb() creates a combined NWB object in memory and returns it, so we can inspect it without saving
    # It saves the nwb only when save_file is provided
    
    if nwb is None:
        print("✗ FAILED: Could not build combined NWB")
        return None
    
    # Extract combined data
    trials_df = nwb.trials.to_dataframe() if nwb.trials else None
    units_df = nwb.units.to_dataframe() if nwb.units else None
    acq_count = len(nwb.acquisition) if nwb.acquisition else 0
    
    # # Print data modalities
    # print(f"DATA MODALITIES:")
    # print(f"{'='*80}")
    # # modality_symbols = {
    # #     'behavior_trials': '🎯',
    # #     'ephys_units': '⚡',
    # #     'lick_times': '👅',
    # #     'reward_times': '🎁',
    # #     'FP': '📊'
    # # }
    # # for mod_name, included in modalities.items():
    # #     symbol = modality_symbols.get(mod_name, '•')
    # #     status = '✓' if included else '✗'
    # #     print(f"  {status} {symbol} {mod_name}")
    
    # print(f"\nACTUAL DATA:")
    # print(f"  Trials: {len(trials_df) if trials_df is not None else 0}")
    # print(f"  Units: {len(units_df) if units_df is not None else 0}")
    # print(f"  Acquisition: {acq_count}")
    
    # Initialize summary
    summary = {
        'session': session_id,
        'modalities': modalities,
        'n_trials': len(trials_df) if trials_df is not None else 0,
        'n_units': len(units_df) if units_df is not None else 0,
        'n_acquisition': acq_count,
        'issues': {}
    }
    
    # Validate trials if they exist
    if session_tbl is not None and trials_df is not None:
        # print(f"\n{'='*80}")
        # print("TRIAL TABLE VALIDATION")
        # print(f"{'='*80}")
        
        trial_map = COLUMN_MAP['behavior_trial_columns']
        trial_desc = COLUMN_DESC['behavior_trial_columns']
        
        missing_cols = []
        wrong_descriptions = []
        value_mismatches = []
        
        for orig_col, mapped_name in trial_map.items():
            if orig_col not in session_tbl.columns:
                continue
            
            if mapped_name not in trials_df.columns:
                missing_cols.append(f"{orig_col} -> {mapped_name}")
                continue
            
            # Check description
            col_idx = trials_df.columns.get_loc(mapped_name)
            actual_desc = nwb.trials.columns[col_idx].description
            expected_desc = trial_desc.get(orig_col, f'Trial column: {orig_col}')
            
            if actual_desc != expected_desc:
                wrong_descriptions.append(f"{mapped_name}")
            
            # Check values (sample first 5 rows)
            for idx in range(min(5, len(session_tbl))):
                source_val = session_tbl[orig_col].iloc[idx]
                nwb_val = trials_df[mapped_name].iloc[idx]
                
                is_equal, diff = compare_values(source_val, nwb_val, mapped_name)
                if not is_equal:
                    value_mismatches.append(f"{mapped_name} row {idx}: {diff}")
                    break
        
        # print(f"Source columns: {len([c for c in trial_map.keys() if c in session_tbl.columns])}")
        # print(f"NWB columns: {len(trials_df.columns)}")
        
        # if not missing_cols:
        #     print(f"✓ All source trial columns included")
        # else:
        #     print(f"✗ Missing columns: {len(missing_cols)}")
            
        # if not wrong_descriptions:
        #     print(f"✓ All trial descriptions match")
        # else:
        #     print(f"⚠ Wrong descriptions: {len(wrong_descriptions)}")
            
        # if not value_mismatches:
        #     print(f"✓ Trial values match (sampled first 5 rows)")
        # else:
        #     print(f"✗ Value mismatches: {len(value_mismatches)}")
        
        summary['issues']['trials'] = {
            'missing_cols': len(missing_cols),
            'wrong_desc': len(wrong_descriptions),
            'value_mismatch': len(value_mismatches)
        }
    
    # Validate units if they exist
    if merged_units is not None and units_df is not None:
        # print(f"\n{'='*80}")
        # print("UNIT TABLE VALIDATION")
        # print(f"{'='*80}")
        
        unit_map_custom = COLUMN_MAP['unit_columns_custom']
        unit_map_ks = COLUMN_MAP['unit_columns_ks']
        
        missing_unit_cols = []
        
        # Check custom columns (excluding predefined NWB columns)
        for orig_col, mapped_name in unit_map_custom.items():
            if orig_col not in merged_units.columns:
                continue
            if 'duplicate as' in mapped_name:
                continue
            if 'similar to' in mapped_name:
                mapped_name = mapped_name.split(';')[0].strip()
            
            # Skip predefined columns - they're handled separately
            if mapped_name in PREDEFINED_COLS:
                continue
            
            if mapped_name not in units_df.columns:
                missing_unit_cols.append(f"{orig_col} -> {mapped_name}")
        
        # Check KS columns (excluding predefined NWB columns)
        for orig_col, mapped_name in unit_map_ks.items():
            if orig_col not in merged_units.columns:
                continue
            
            # Skip predefined columns - they're handled separately
            if mapped_name in PREDEFINED_COLS:
                continue
            
            if mapped_name not in units_df.columns:
                missing_unit_cols.append(f"{orig_col} -> {mapped_name}")
        
        # print(f"Source columns: {len(merged_units.columns)}")
        # print(f"NWB columns: {len(units_df.columns)}")
        
        # if not missing_unit_cols:
        #     print(f"✓ All source unit columns included (excluding predefined NWB columns)")
        # else:
        #     print(f"✗ Missing unit columns: {len(missing_unit_cols)}")
        #     for col in missing_unit_cols[:5]:
        #         print(f"    - {col}")
        
        summary['issues']['units'] = {
            'missing_cols': len(missing_unit_cols)
        }
    
    # # Summary
    # print(f"\n{'='*80}")
    # print("SUMMARY")
    # print(f"{'='*80}")
    
    total_issues = sum(sum(v.values()) if isinstance(v, dict) else 0 
                      for v in summary['issues'].values())
    
    # if total_issues == 0:
    #     print(f"✓ ALL CHECKS PASSED")
    #     summary['status'] = 'PASSED'
    # else:
    #     print(f"⚠ Total issues: {total_issues}")
    #     summary['status'] = 'ISSUES'
    
    # Print summary dictionary
    # print(f"\nSUMMARY DICT:")
    # print(json.dumps(summary, indent=2, default=str))
    
    return summary


In [3]:

# Test sessions (including one with FP data)
example_sessions = [
    'behavior_754897_2025-03-13_11-20-42',
    'behavior_ZS061_2021-04-08_18-01-30',
    'behavior_754898_2025-01-01_20-40-03',  # Has FP data
]

all_summaries = []
for session in example_sessions:
    try:
        summary = test_session_comprehensive(session, data_type='curated')
        if summary:
            all_summaries.append(summary)
    except Exception as e:
        print(f"✗ Exception: {e}")
        import traceback
        traceback.print_exc()

# print(f"\n\n{'#'*80}")
# print("ALL SESSIONS SUMMARY")
# print(f"{'#'*80}\n")
# print(json.dumps(all_summaries, indent=2, default=str))


################################################################################
# behavior_754897_2025-03-13_11-20-42
################################################################################


################################################################################
# behavior_ZS061_2021-04-08_18-01-30
################################################################################


################################################################################
# behavior_754898_2025-01-01_20-40-03
################################################################################



No custom unit table found for behavior_754898_2025-01-01_20-40-03
No custom unit table found for behavior_754898_2025-01-01_20-40-03
No unit tables to merge for behavior_754898_2025-01-01_20-40-03 - will create NWB with behavior/acquisition only


No unit table found for behavior_754898_2025-01-01_20-40-03 in curated data.
No unit table found for behavior_754898_2025-01-01_20-40-03 in curated data.


In [4]:
# make a table of the results, each key in the summary dict becomes a column, and each session is a row
summary_df = pd.DataFrame(all_summaries)
# put modalities into separate columns
modalities_df = summary_df['modalities'].apply(pd.Series)
# put issues into separate columns
issues_df = summary_df['issues'].apply(lambda x: pd.Series({
    'trials_missing_cols': x['trials']['missing_cols'] if 'trials' in x else np.nan,
    'trials_wrong_desc': x['trials']['wrong_desc'] if 'trials' in x else np.nan,
    'trials_value_mismatch': x['trials']['value_mismatch'] if 'trials' in x else np.nan,
    'units_missing_cols': x['units']['missing_cols'] if 'units' in x else np.nan,
}))
# merge with summary_df
final_df = pd.concat([summary_df.drop(columns=['modalities']), modalities_df], axis=1)
final_df = pd.concat([final_df.drop(columns=['issues']), issues_df], axis=1)
print(final_df)

                               session  n_trials  n_units  n_acquisition  \
0  behavior_754897_2025-03-13_11-20-42       564      269              4   
1   behavior_ZS061_2021-04-08_18-01-30       386        2              4   
2  behavior_754898_2025-01-01_20-40-03       542        0             53   

   behavior_trials  ephys_units  lick_times  reward_times     FP beh_version  \
0             True         True        True          True  False   processed   
1             True         True        True          True  False         raw   
2             True        False        True          True   True         raw   

                        nwb_created nwb_saved  trials_missing_cols  \
0  2026-06-11T09:29:57.766467+00:00      None                  0.0   
1  2026-06-11T09:30:11.140239+00:00      None                  0.0   
2  2026-06-11T09:30:13.261505+00:00      None                  0.0   

   trials_wrong_desc  trials_value_mismatch  units_missing_cols  
0                0.0       

In [5]:
# load all used sessions
dfs = [pd.read_csv(CAPSULE_ROOT + '/code/data_management/session_assets.csv'),
        pd.read_csv(CAPSULE_ROOT + '/code/data_management/hopkins_session_assets.csv'),
        pd.read_csv(CAPSULE_ROOT + '/code/data_management/hopkins_FP_session_assets.csv')]
df = pd.concat(dfs)
# remove empty rows
df = df.dropna(subset=['session_id'])
session_list = df['session_id'].values.tolist()